# Practical 5: Continuous Bag of Words (CBOW)

**What we are doing.** We turn each word into a small vector of 10 numbers (an **embedding**) and learn those vectors through a guessing game: show the **2 words on each side** and predict the missing **middle word**. Words used in similar places end up with similar vectors.

```
"deep learning ___ known as" → 4 word ids → Embedding (4 x 10) → average (10)
                             → Dense(vocab, softmax) → "also"
```
**Skeleton:** imports → load → preprocess → **model** → compile → fit → evaluate → plot/predict

> **Run locally in Jupyter?** Install once: `%pip install tensorflow numpy pandas matplotlib scikit-learn`, then *Kernel → Restart Kernel*.
> **In Colab:** *Runtime → Change runtime type → T4 GPU*. Everything is preinstalled.

In [ ]:
import re
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

text = """Deep learning (also known as deep structured learning) is part of a broader family of machine learning methods based on artificial neural networks with representation learning. Learning can be supervised, semi-supervised or unsupervised. Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning, recurrent neural networks, convolutional neural networks and Transformers have been applied to fields including computer vision, speech recognition, natural language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, material inspection and board game programs, where they have produced results comparable to and in some cases surpassing human expert performance."""

## a. Data preparation: build the vocabulary
- Split into sentences at `.`, lowercase, and replace every non a–z character with a space (`re.sub('[^a-z]', ' ', ...)`), then split into words.
- `vocab` = unique words, sorted. `word_to_index` maps word → id, and `index_to_word` maps id → word.
- Result: **74 words** in the vocabulary.

In [ ]:
sentences = []
for s in text.split('.'):
    words = re.sub('[^a-z]', ' ', s.lower()).split()
    if words:
        sentences.append(words)

vocab = sorted({w for s in sentences for w in s})
word_to_index = {w: i for i, w in enumerate(vocab)}
index_to_word = {i: w for w, i in word_to_index.items()}
vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)

## b. Generate training data: context → target
For every word that has 2 neighbours on each side: `X` gets the 4 context ids and `Y` gets the middle id. That gives **88 pairs**.

In [ ]:
window = 2
X, Y = [], []
for s in sentences:
    ids = [word_to_index[w] for w in s]
    for i in range(window, len(ids) - window):
        X.append(ids[i - window:i] + ids[i + 1:i + window + 1])
        Y.append(ids[i])
X, Y = np.array(X), np.array(Y)
for i in range(5):
    print([index_to_word[j] for j in X[i]], '->', index_to_word[Y[i]])

## c. Model and train
- **Embedding(74, 10)**: a lookup table with one learnable row of 10 numbers per word. After training, *this table is the word embeddings*.
- **GlobalAveragePooling1D**: averages the 4 context vectors. That's the "bag": word order doesn't matter.
- **Dense(vocab, softmax)**: one probability per word.
- Adam with lr 0.05 and 200 epochs, because the dataset is tiny.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(2 * window,)),
    layers.Embedding(vocab_size, 10),                # each word id -> a vector of 10 numbers
    layers.GlobalAveragePooling1D(),                 # average the 4 context vectors
    layers.Dense(vocab_size, activation='softmax'),  # probability of every word being the middle one
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X, Y, epochs=200, verbose=0)
print("Final training accuracy:", history.history['accuracy'][-1])

## d. Output: predict the missing middle word

In [ ]:
tests = ["known as structured learning",
         "transformers have applied to",
         "where they produced results",
         "cases surpassing expert performance"]
for t in tests:
    ids = np.array([[word_to_index[w] for w in t.split()]])
    print(t, '->', index_to_word[model.predict(ids, verbose=0).argmax()])

# The learned word vectors are the Embedding layer's weights
embeddings = model.layers[0].get_weights()[0]
print("Embedding matrix shape:", embeddings.shape)   # (vocab_size, 10)

---
## Understand it: poke cells

In [ ]:
# A. Training pairs as ids and as words
print(X[:3], Y[:3])
for i in range(3): print([index_to_word[j] for j in X[i]], '->', index_to_word[Y[i]])

# B. One word's embedding: 10 learned numbers
print(model.layers[0](np.array([[word_to_index['deep']]])).numpy())

# C. Top 3 guesses for one context
p = model.predict(np.array([[word_to_index[w] for w in "known as structured learning".split()]]), verbose=0)[0]
print([(index_to_word[i], round(float(p[i]), 3)) for i in p.argsort()[::-1][:3]])

---
## Break it
Set `epochs=5`. Accuracy stays near 0 and the predictions are junk, because the vectors start random and only become meaningful through training. Also, 88 pairs is far too few for meaningful "similar words". Real word2vec trains on millions of words.

---
## Viva answers
**Q1. CBOW vs Skip-gram?** CBOW predicts the middle word from its context. Skip-gram predicts the context from the middle word. CBOW is faster, and Skip-gram handles rare words better.

**Q2. What does the Embedding layer hold?** A (vocab_size × 10) table with one learnable vector per word. After training, those weights *are* the embeddings.

**Q3. Why average the context vectors?** CBOW treats the context as a bag (order doesn't matter), so the 4 vectors are averaged into one before the softmax over the vocabulary.

> **Classmate's code** uses `Tokenizer` and `Embedding(input_length=...)`, both removed in Keras 3 (current Colab), so it can fail. It also adds Dense 256/512 layers that aren't part of CBOW.